# Notebook 03 — Embedding Visualization

Generates all **24 embedding plots** (4 types × 6 models):
- **Type 1**: Ground truth class coloring (41-color palette) + Silhouette Score
- **Type 2**: Error overlay (misclassified nodes as red ×)
- **Type 3**: Node degree overlay (viridis colormap)
- **Type 4**: 2×3 cross-model comparison grid

Also generates GAT/GATv2 attention visualizations (D4/E1 analysis).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Image

from reddit_gnn.config import (
    DEVICE, RESULTS_ROOT, CHECKPOINTS_DIR, NUM_CLASSES, NUM_FEATURES,
    DEFAULT_HPARAMS, SGC_DIR, PREPROCESSED
)
from reddit_gnn.data.normalize import load_normalized_data
from reddit_gnn.analysis.visualisation import (
    stratified_sample, compute_tsne, compute_umap,
    compute_embedding_quality_metrics,
    plot_type1_ground_truth, plot_type2_error_overlay,
    plot_type3_degree_overlay, plot_type4_cross_model_grid,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

SAVE_DIR = os.path.join(RESULTS_ROOT, 'figures', '03_visualisation')
os.makedirs(SAVE_DIR, exist_ok=True)
N_PER_CLASS = 500   # 500 × 41 = 20,500 nodes total

print('Loading data...')
data, _, _ = load_normalized_data()
print(f'Loaded: {data.num_nodes:,} nodes, {data.num_edges:,} edges')

## 1. Helper — Load Checkpoint & Extract Embeddings

In [ ]:
def load_model_embeddings(model_name):
    ckpt = os.path.join(CHECKPOINTS_DIR, model_name, 'baseline', 'default', 'seed0', 'best_model.pt')
    if not os.path.exists(ckpt):
        print(f'  ⚠️  No checkpoint for {model_name}: {ckpt}')
        return None, None
    
    hp = DEFAULT_HPARAMS[model_name]
    if model_name == 'graphsage':
        from reddit_gnn.models.graphsage import GraphSAGE
        model = GraphSAGE(NUM_FEATURES, hp['hidden'], NUM_CLASSES, aggregator=hp['aggregator']).to(DEVICE)
    elif model_name == 'graphsaint':
        from reddit_gnn.models.graphsaint import GraphSAINTNet
        model = GraphSAINTNet(NUM_FEATURES, hp['hidden'], NUM_CLASSES).to(DEVICE)
    elif model_name == 'sgc':
        from reddit_gnn.models.sgc import SGC
        model = SGC(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
    elif model_name == 'gat':
        from reddit_gnn.models.gat import GAT
        model = GAT(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    elif model_name == 'gatv2':
        from reddit_gnn.models.gatv2 import GATv2
        model = GATv2(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    elif model_name == 'cluster_gcn':
        from reddit_gnn.models.cluster_gcn import ClusterGCN
        model = ClusterGCN(NUM_FEATURES, hp['hidden'], NUM_CLASSES).to(DEVICE)
    else:
        return None, None
    
    model.load_state_dict(torch.load(ckpt, weights_only=False))
    model.eval()
    
    with torch.no_grad():
        data_dev = data.to(DEVICE)
        if model_name == 'sgc':
            K = hp.get('K', 2)
            X_K = torch.load(os.path.join(SGC_DIR, f'reddit_sgc_K{K}.pt'), weights_only=False).to(DEVICE)
            embeddings = X_K.cpu()
            out = model(X_K)
        else:
            embeddings = model.encode(data_dev.x, data_dev.edge_index).cpu()
            out = model(data_dev.x, data_dev.edge_index)
        preds = out.argmax(1).cpu()
    
    return embeddings, preds

print('Helper defined.')

## 2. Generate All Plots for Each Model

In [ ]:
from torch_geometric.utils import degree as pyg_degree

MODEL_NAMES   = ['graphsage','graphsaint','sgc','gat','gatv2','cluster_gcn']
DISPLAY_NAMES = ['GraphSAGE','GraphSAINT','SGC','GAT','GATv2','ClusterGCN']

ALL_EMB_2D, ALL_LABELS, ALL_ACCS, VALID_NAMES = [], [], [], []

deg = pyg_degree(data.edge_index[0], num_nodes=data.num_nodes).numpy()
test_mask = data.test_mask
test_indices = torch.where(test_mask)[0]

for model_name, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    print(f'\n=== {display} ===')
    embeddings, preds = load_model_embeddings(model_name)
    if embeddings is None:
        continue
    
    y_test = data.y[test_mask]
    vis_idx = stratified_sample(y_test, test_indices, n_per_class=N_PER_CLASS)
    vis_emb = embeddings[vis_idx].numpy()
    vis_labels = data.y[vis_idx].numpy()
    vis_preds  = preds[vis_idx].numpy()
    vis_deg    = deg[vis_idx.numpy()]
    
    # True test accuracy
    preds_test = preds[test_mask].numpy()
    test_acc   = (preds_test == y_test.numpy()).mean()
    
    for method_name, method_fn in [('tsne', compute_tsne), ('umap', compute_umap)]:
        print(f'  Computing {method_name.upper()}...')
        try:
            emb_2d = method_fn(vis_emb)
        except Exception as e:
            print(f'  {method_name} failed: {e}'); continue
        
        q = compute_embedding_quality_metrics(emb_2d, vis_labels)
        print(f'  Silhouette={q["silhouette"]:.4f} | DB={q["davies_bouldin"]:.4f} | CH={q["calinski_harabasz"]:.1f}')
        
        save_pref = os.path.join(SAVE_DIR, model_name, method_name)
        plot_type1_ground_truth(emb_2d, vis_labels, f'{display} ({method_name.upper()})', test_acc,
                                save_path=os.path.join(save_pref, 'type1_ground_truth.png'))
        plot_type2_error_overlay(emb_2d, vis_labels, vis_preds, f'{display} ({method_name.upper()})',
                                 save_path=os.path.join(save_pref, 'type2_error_overlay.png'))
        plot_type3_degree_overlay(emb_2d, vis_deg, f'{display} ({method_name.upper()})',
                                  save_path=os.path.join(save_pref, 'type3_degree_overlay.png'))
        print(f'  → Saved 3 plots to {save_pref}')
        
        if method_name == 'tsne':  # collect for grid
            ALL_EMB_2D.append(emb_2d); ALL_LABELS.append(vis_labels)
            ALL_ACCS.append(test_acc); VALID_NAMES.append(display)

print('\nDone processing all models.')

## 3. Type 4 — Cross-Model Comparison Grid (2×3)

In [ ]:
if len(ALL_EMB_2D) >= 2:
    save_path = os.path.join(SAVE_DIR, 'type4_cross_model_grid.png')
    plot_type4_cross_model_grid(ALL_EMB_2D, ALL_LABELS, VALID_NAMES, ALL_ACCS, save_path=save_path)
    print(f'Saved Type 4 grid: {save_path}')
    display(Image(save_path))
else:
    print('Not enough models loaded for cross-model grid.')

## 4. Attention Visualization — GAT/GATv2 (D4/E1)

In [ ]:
# Load GAT and GATv2 baselines for attention comparison
from reddit_gnn.analysis.attention_analysis import (
    extract_attention_weights, compute_attention_entropy,
    hub_concentration_test, homophily_aware_attention
)

for model_name, display in [('gat','GAT'), ('gatv2','GATv2')]:
    ckpt = os.path.join(CHECKPOINTS_DIR, model_name, 'baseline', 'default', 'seed0', 'best_model.pt')
    if not os.path.exists(ckpt):
        print(f'  {display}: No checkpoint found — skipping')
        continue
    
    hp = DEFAULT_HPARAMS[model_name]
    if model_name == 'gat':
        from reddit_gnn.models.gat import GAT
        model = GAT(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    else:
        from reddit_gnn.models.gatv2 import GATv2
        model = GATv2(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    
    model.load_state_dict(torch.load(ckpt, weights_only=False))
    
    # Sample 50 test nodes (mix of high/low degree)
    from torch_geometric.utils import degree as pyg_deg
    ddeg = pyg_deg(data.edge_index[0], num_nodes=data.num_nodes)
    tidx = torch.where(data.test_mask)[0]
    sample = tidx[ddeg[tidx].argsort(descending=True)[:25]].tolist() + \
             tidx[ddeg[tidx].argsort()[:25]].tolist()
    
    print(f'\n{display} — Attention Analysis:')
    try:
        attn = extract_attention_weights(model, data, DEVICE, sample)
        entropies = compute_attention_entropy(attn)
        
        # Entropy bar chart for sampled nodes
        norm_ents = [v['normalized_entropy'] for v in entropies.values()]
        fig, ax = plt.subplots(figsize=(12,4))
        ax.bar(range(len(norm_ents)), sorted(norm_ents), color='steelblue', alpha=0.8)
        ax.axhline(0.9, color='red', ls='--', label='0.9 (near-uniform threshold)')
        ax.set_xlabel('Node (sorted by entropy)', fontsize=11)
        ax.set_ylabel('Normalized Entropy H(α)', fontsize=11)
        ax.set_title(f'{display}: Attention Entropy Distribution ({len(norm_ents)} nodes)', fontsize=12)
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, f'{model_name}_attention_entropy.png'), dpi=150)
        plt.show()
        
        hub_concentration_test(attn, top_k=10)
        homophily_aware_attention(attn, data)
    except Exception as e:
        print(f'  Attention analysis failed: {e}')

## 5. Oversmoothing Visualization — Embedding Variance Per Layer

In [ ]:
from reddit_gnn.analysis.oversmoothing import compute_embedding_variance_per_layer, oversmoothing_summary

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_configs = [('graphsage','GraphSAGE'), ('gat','GAT'), ('gatv2','GATv2')]

for ax, (model_name, display) in zip(axes, model_configs):
    ckpt = os.path.join(CHECKPOINTS_DIR, model_name, 'baseline', 'default', 'seed0', 'best_model.pt')
    if not os.path.exists(ckpt):
        ax.set_title(f'{display} (no checkpoint)')
        continue
    
    hp = DEFAULT_HPARAMS[model_name]
    if model_name == 'graphsage':
        from reddit_gnn.models.graphsage import GraphSAGE
        model = GraphSAGE(NUM_FEATURES, hp['hidden'], NUM_CLASSES, aggregator=hp['aggregator']).to(DEVICE)
    elif model_name == 'gat':
        from reddit_gnn.models.gat import GAT
        model = GAT(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    else:
        from reddit_gnn.models.gatv2 import GATv2
        model = GATv2(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, weights_only=False))
    
    try:
        stats = compute_embedding_variance_per_layer(model, data, DEVICE)
        layers = range(1, len(stats['variance'])+1)
        ax.plot(layers, stats['variance'], 'b-o', label='Variance')
        ax.set_xlabel('Layer'); ax.set_ylabel('Avg Variance', color='blue')
        ax2 = ax.twinx()
        ax2.plot(layers, stats['intra_class_sim'], 'g--s', label='Intra-class sim')
        ax2.plot(layers, stats['inter_class_sim'], 'r--^', label='Inter-class sim')
        ax2.set_ylabel('Cosine Similarity')
        ax.set_title(f'{display} — Oversmoothing')
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1+lines2, labels1+labels2, fontsize=8)
    except Exception as e:
        ax.set_title(f'{display} — Error: {e}')

plt.suptitle('Oversmoothing: Variance & Cosine Similarity Per Layer', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'oversmoothing_variance.png'), dpi=150)
plt.show()